# CRCNS hc-6 `.mat` import starter (Neuropy-compatible)

Minimal starter notebook for loading locally downloaded CRCNS hc-6 MATLAB files, inspecting structure, and creating Neuropy-compatible objects where possible.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Prefer repository-style helper if available
import_mat_file = None
for _import_stmt in (
    'from pyphoplacecellanalysis.PhoPositionalData.load_exported import import_mat_file',
    'from neuropy.utils.load_exported import import_mat_file',
):
    try:
        exec(_import_stmt)
        break
    except Exception:
        pass

if import_mat_file is None:
    from scipy.io import loadmat
    def import_mat_file(mat_import_file, **kwargs):
        return loadmat(mat_import_file, squeeze_me=False, struct_as_record=False)

# Optional Neuropy object constructors
try:
    from neuropy.core import Position, Epoch, Neurons
    from neuropy.core.session.dataSession import DataSession
except Exception:
    Position = Epoch = Neurons = DataSession = None


In [ ]:
# Edit this to your local CRCNS hc-6 extraction root
base_dir = Path('/path/to/local/crcns/hc-6')
assert base_dir.exists(), f'Update base_dir first: {base_dir}'

mat_files = sorted(base_dir.rglob('*.mat'))
print(f'Found {len(mat_files)} .mat files under {base_dir}')
for p in mat_files[:20]:
    print(' -', p.relative_to(base_dir))


In [ ]:
# Inspect keys in one file (adjust index if needed)
example_file = mat_files[0]
example_data = import_mat_file(example_file)
example_keys = [k for k in example_data.keys() if not str(k).startswith('__')]
print('Example file:', example_file)
print('Top-level keys:', example_keys)


In [ ]:
def _pick_file(candidates):
    for pat in candidates:
        hits = sorted(base_dir.rglob(pat))
        if hits:
            return hits[0]
    return None

def _extract_struct_item(arr, field):
    if arr is None:
        return None
    if isinstance(arr, np.ndarray) and getattr(arr.dtype, 'names', None) and field in arr.dtype.names:
        out = arr[field]
        return np.asarray(out).reshape(-1)
    if isinstance(arr, np.ndarray) and arr.size > 0 and hasattr(arr.flat[0], field):
        out = getattr(arr.flat[0], field)
        return np.asarray(out).reshape(-1)
    return None

position_file = _pick_file(['*position*.mat', '*pos*.mat', '*vt*.mat'])
spikes_file = _pick_file(['*spike*.mat', '*spikes*.mat', '*cell*.mat'])
epochs_file = _pick_file(['*epoch*.mat', '*behavior*.mat', '*states*.mat'])
print('position_file:', position_file)
print('spikes_file:  ', spikes_file)
print('epochs_file:  ', epochs_file)


In [ ]:
# Convert position -> Neuropy Position or dict
position_obj = None
if position_file is not None:
    pos = import_mat_file(position_file)
    pos_keys = [k for k in pos.keys() if not str(k).startswith('__')]
    pos_key = next((k for k in pos_keys if 'pos' in k.lower() or 'vt' in k.lower()), pos_keys[0] if pos_keys else None)
    pos_data = pos.get(pos_key) if pos_key else None

    t = _extract_struct_item(pos_data, 'tt')
    if t is None:
        t = _extract_struct_item(pos_data, 't')
    x = _extract_struct_item(pos_data, 'x')
    y = _extract_struct_item(pos_data, 'y')
    if x is None:
        x = _extract_struct_item(pos_data, 'xx')
        if x is not None and x.ndim > 1:
            x = x.reshape(-1)
    if y is None:
        y = _extract_struct_item(pos_data, 'yy')
        if y is not None and y.ndim > 1:
            y = y.reshape(-1)

    if t is not None and x is not None and y is not None:
        t = np.asarray(t, dtype=float).reshape(-1)
        x = np.asarray(x, dtype=float).reshape(-1)
        y = np.asarray(y, dtype=float).reshape(-1)
        n = min(len(t), len(x), len(y))
        t, x, y = t[:n], x[:n], y[:n]
        t = t - t[0]
        if np.nanmax(t) > 1e4:  # common microsecond timestamps
            t = t / 1e6
        position_obj = Position.from_separate_arrays(t, x, y) if Position is not None else {'time': t, 'x': x, 'y': y}

print('position_obj type:', type(position_obj).__name__ if position_obj is not None else None)


In [ ]:
# Convert spikes -> Neuropy Neurons or dataframe
neurons_obj = None
spikes_df = None
if spikes_file is not None:
    spk = import_mat_file(spikes_file)
    spk_keys = [k for k in spk.keys() if not str(k).startswith('__')]
    spk_key = next((k for k in spk_keys if 'spike' in k.lower()), spk_keys[0] if spk_keys else None)
    spk_data = spk.get(spk_key) if spk_key else None

    t = _extract_struct_item(spk_data, 't')
    aclu = _extract_struct_item(spk_data, 'aclu')
    qclu = _extract_struct_item(spk_data, 'qclu')
    shank = _extract_struct_item(spk_data, 'shank')

    if t is not None and aclu is not None:
        t = np.asarray(t, dtype=float).reshape(-1)
        if np.nanmax(t) > 1e4:
            t = t / 1e6
        aclu = np.asarray(aclu).reshape(-1).astype(int)
        n = min(len(t), len(aclu))
        t, aclu = t[:n], aclu[:n]

        spikes_df = pd.DataFrame({'t': t, 'aclu': aclu})
        if qclu is not None:
            spikes_df['qclu'] = np.asarray(qclu).reshape(-1)[:n]
        if shank is not None:
            spikes_df['shank'] = np.asarray(shank).reshape(-1)[:n]

        if Neurons is not None:
            grouped = spikes_df.groupby('aclu', sort=True)
            neuron_ids = [int(i) for i in grouped.indices.keys()]
            spiketrains = np.array([grouped.get_group(i)['t'].to_numpy() for i in neuron_ids], dtype='object')
            t_stop = float(np.nanmax(t))
            shank_ids = np.array([grouped.get_group(i)['shank'].iloc[0] if 'shank' in grouped.get_group(i) else np.nan for i in neuron_ids])
            neurons_obj = Neurons(spiketrains, t_stop=t_stop, t_start=0.0, neuron_ids=neuron_ids, shank_ids=shank_ids)

print('spikes_df shape:', None if spikes_df is None else spikes_df.shape)
print('neurons_obj type:', type(neurons_obj).__name__ if neurons_obj is not None else None)


In [ ]:
# Convert epochs -> Neuropy Epoch or dataframe
epochs_obj = None
epochs_df = None
if epochs_file is not None:
    ep = import_mat_file(epochs_file)
    ep_keys = [k for k in ep.keys() if not str(k).startswith('__')]
    ep_key = next((k for k in ep_keys if 'epoch' in k.lower() or 'state' in k.lower()), ep_keys[0] if ep_keys else None)
    ep_data = ep.get(ep_key) if ep_key else None

    start = _extract_struct_item(ep_data, 'start')
    stop = _extract_struct_item(ep_data, 'stop')
    label = _extract_struct_item(ep_data, 'label')

    if start is not None and stop is not None:
        start = np.asarray(start, dtype=float).reshape(-1)
        stop = np.asarray(stop, dtype=float).reshape(-1)
        n = min(len(start), len(stop))
        start, stop = start[:n], stop[:n]
        if np.nanmax(stop) > 1e4:
            start, stop = start / 1e6, stop / 1e6

        if label is None:
            label = np.array([f'epoch_{i}' for i in range(n)])
        else:
            label = np.asarray(label).reshape(-1)[:n]
        epochs_df = pd.DataFrame({'start': start, 'stop': stop, 'label': label.astype(str)})
        epochs_obj = Epoch(epochs=epochs_df) if Epoch is not None else epochs_df

print('epochs_df shape:', None if epochs_df is None else epochs_df.shape)
print('epochs_obj type:', type(epochs_obj).__name__ if epochs_obj is not None else None)


In [ ]:
# Optional assembly into a DataSession-like container
session = None
if DataSession is not None:
    session = DataSession()
    if position_obj is not None:
        session.position = position_obj
    if epochs_obj is not None:
        session.epochs = epochs_obj
    if neurons_obj is not None:
        session.neurons = neurons_obj

print('session type:', type(session).__name__ if session is not None else None)
if session is None:
    session = {'position': position_obj, 'epochs': epochs_obj, 'neurons': neurons_obj}

session

If your hc-6 filenames differ, only edit the `base_dir` cell and (optionally) the filename patterns in `_pick_file(...)`.